# 🏗️ Módulo 4: Inversión Patrimonial en Finca Raíz Colombiana
**Responsable OED:** Jesi (*Administradora de Empresas — Magíster en Analítica*)  
**Caso de Uso:** `CU-04 — Evaluación Analítica de Inversión en Finca Raíz (Vivienda Propia o Renta)`

---

### 🎯 Objetivo del Notebook
Este notebook ejecuta de forma **100% autónoma** (usando el *Usuario Semilla* de `config/config.yaml`) toda la analítica inmobiliaria y regulatoria colombiana del **Módulo 4**:
1. **Clasificación Regulatoria y Plan Sobre Planos:** Segmentación automática **VIP ($\le 90\text{ SMMLV}$)**, **VIS ($\le 150\text{ SMMLV}$)** y **No VIS ($> 150\text{ SMMLV}$)**, subsidios de Caja de Compensación Familiar ($\le 4\text{ SMMLV}$) y modelación del encargo fiduciario ($18-36$ meses) integrando **Cuenta AFC (Art. 126-4 E.T.)**, cesantías y primas.
2. **Ley 546 de 1999 (Regla del 30%) y Pesos vs. UVR:** Comparación mes a mes de los sistemas de amortización autorizados por la SFC (**Cuota Fija en Pesos**, **Abono Constante a Capital** y **UVR indexada al IPC DANE**), cálculo exacto de brecha de cuota inicial/ingreso y alerta `FA-01` de amortización lenta en UVR.
3. **Evaluación Financiera Nivel Maestría (*Buy-to-Rent* y Ley 820 de 2003):** Cálculo de **NOI**, **Cap Rate Neto**, **Cash-on-Cash**, **$VPN$** y **$TIR$** neta de Ganancia Ocasional ($15\%$, Ley 2277 de 2022), comparando comprar inmueble frente a arrendar e invertir en un **FIC Inmobiliario / Portafolio Financiero**.

In [ ]:
import sys
from pathlib import Path

# Asegurar que la raíz del repositorio esté en sys.path sin importar desde dónde se abra Jupyter
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.finance import cargar_configuracion
from src.mod4_finca_raiz import (
    clasificar_segmento_vivienda_colombia,
    estimar_costo_oportunidad_por_perfil_oed,
    evaluar_inversion_finca_raiz,
    simular_plan_cuota_inicial_planos,
    simular_sistemas_amortizacion_ley_546,
)
from src.ui_mod4_finca_raiz import generar_resumen_ejecutivo_ciudadano

cfg = cargar_configuracion()
semilla = cfg["perfil_semilla_taller"]
macro = cfg["parametros_macro_colombia"]

print("✅ Configuración OED cargada correctamente:")
print(f"   • Vigencia: {macro['anio_vigencia']} | SMMLV: ${macro['smmlv_cop']:,.0f} COP | UVR: ${macro['uvr_valor_cop']:,.2f} COP | IPC DANE: {macro['ipc_anual_dane']*100:.1f}%")
print(f"   • Usuario Semilla: {semilla['nombre_alias']} | Ingreso Bruto: ${semilla['ingreso_bruto_mensual_cop']:,.0f} COP")

---
## 1. Paso 1: Clasificación VIP / VIS / No VIS y Plan de Cuota Inicial Sobre Planos
Evaluamos tres tipologías de vivienda en Colombia y simulamos el flujo de caja del **encargo fiduciario a 24 meses** para el **Caso Maestro OED** (Apartamento VIS de **$200.000.000 COP**), combinando ahorro mensual en **Cuenta AFC**, primas semestrales y cesantías anuales.

In [ ]:
# Comparativa regulatoria de segmentos inmobiliarios en Colombia
casos_segmento = [
    ("Apartamento VIP Social", 120_000_000.0, 4_200_000.0),
    ("Apartamento VIS (Caso Maestro OED)", 200_000_000.0, 9_415_000.0),
    ("Apartamento No VIS Familiar", 340_000_000.0, 9_415_000.0),
]

filas_seg = []
for nombre, valor, ingreso in casos_segmento:
    c = clasificar_segmento_vivienda_colombia(valor, ingreso, config=cfg)
    filas_seg.append({
        "Proyecto": nombre,
        "Valor Comercial (COP)": f"${valor:,.0f}",
        "Valor (SMMLV)": c["valor_inmueble_smmlv"],
        "Clasificación": c["segmento_vivienda"],
        "Cuota Inicial Mín. Legal": f"{c['cuota_inicial_min_legal_pct']*100:.0f}%",
        "Financiación Máx. LTV": f"{c['financiacion_max_ltv_pct']:.0f}%",
        "Subsidio Potencial CCF": f"${c['subsidio_ccf_potencial_cop']:,.0f}",
    })

pd.DataFrame(filas_seg)

In [ ]:
# Simulación del Encargo Fiduciario a 24 meses (Sobre Planos) con Cuenta AFC + Cesantías + Primas
plan_planos = simular_plan_cuota_inicial_planos(
    valor_inmueble_cop=200_000_000.0,
    ingreso_familiar_neto_cop=9_415_000.0,
    porcentaje_cuota_inicial=0.30,
    plazo_entrega_planos_meses=24,
    cesantias_y_primas_anuales_cop=5_500_000.0,
    ahorro_previo_afc_cop=8_000_000.0,
    config=cfg,
)

df_fiducia = pd.DataFrame(plan_planos["cronograma_fiduciario_mensual"])

print(f"🔑 Meta Cuota Inicial (30%): ${plan_planos['cuota_inicial_total_cop']:,.0f} COP")
print(f"💰 Aporte Mensual en Cuenta AFC: ${plan_planos['cuota_inicial_mensual_cop']:,.0f} COP ({plan_planos['esfuerzo_mensual_cuota_inicial_pct']}% del ingreso familiar)")
print(f"🏛️ Ahorro Tributario Estimado vía AFC (Art. 126-4 E.T.): ${plan_planos['ahorro_tributario_afc_total_cop']:,.0f} COP")
print(f"📈 Rendimientos Ganados en Encargo Fiduciario: ${plan_planos['rendimientos_encargo_fiduciario_cop']:,.0f} COP")

fig_planos = make_subplots(specs=[[{"secondary_y": True}]])
fig_planos.add_trace(
    go.Bar(
        x=df_fiducia["mes"],
        y=df_fiducia["cuota_mensual_afc_cop"],
        name="Aporte Mensual Salarial (AFC)",
        marker_color="#1f77b4",
    ),
    secondary_y=False,
)
fig_planos.add_trace(
    go.Bar(
        x=df_fiducia["mes"],
        y=df_fiducia["abono_extra_cesantias_primas_subsidio_cop"],
        name="Abonos Extras (Primas Jun/Dic + Cesantías)",
        marker_color="#2ca02c",
    ),
    secondary_y=False,
)
fig_planos.add_trace(
    go.Scatter(
        x=df_fiducia["mes"],
        y=df_fiducia["saldo_acumulado_encargo_cop"],
        name="Saldo Acumulado Encargo Fiduciario",
        mode="lines+markers",
        line=dict(color="#ff7f0e", width=3),
    ),
    secondary_y=True,
)
fig_planos.update_layout(
    barmode="stack",
    title="Plan de Pago de Cuota Inicial Sobre Planos (24 Meses — Encargo Fiduciario + AFC)",
    xaxis_title="Mes del Encargo Fiduciario",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_planos.update_yaxes(title_text="Aportes Mensuales (COP)", secondary_y=False)
fig_planos.update_yaxes(title_text="Saldo Acumulado en Fiducia (COP)", secondary_y=True)
fig_planos.show()

---
## 2. Paso 2: Simulador Ley 546 de 1999 (Regla del 30%) y Pesos vs. UVR
Según la **Ley 546 de 1999 (Ley de Vivienda)**, la primera cuota del crédito hipotecario (incluyendo seguros obligatorios de vida e incendio/terremoto) **no puede superar el 30% del ingreso familiar mensual**.

A continuación comparamos los **3 sistemas de amortización autorizados por la SFC** sobre un crédito de **$140.000.000 COP** a 15 años:
1. **Cuota Fija en Pesos** (estabilidad nominal absoluta).
2. **Abono Constante a Capital en Pesos** (cuota inicial más exigente, pero menor pago total de intereses).
3. **Cuota Constante en UVR** (primera cuota más baja, pero expuesta a la inflación IPC DANE — **Alerta `FA-01`**).

In [ ]:
ingreso_hogar_demo = 9_415_000.0
tope_ley_546 = ingreso_hogar_demo * 0.30

amort = simular_sistemas_amortizacion_ley_546(
    monto_credito_cop=140_000_000.0,
    plazo_credito_anios=15,
    tasa_ea_pesos=0.115,
    tasa_ea_uvr_mas_spread=0.065,
    config=cfg,
)

df_pesos_fija = pd.DataFrame(amort["tabla_pesos_cuota_fija"])
df_pesos_const = pd.DataFrame(amort["tabla_pesos_abono_constante"])
df_uvr = pd.DataFrame(amort["tabla_uvr_ciclica"])

resumen_sistemas = pd.DataFrame([
    {
        "Sistema SFC": "1. Pesos — Cuota Fija (Francés)",
        "Primera Cuota (COP)": f"${amort['primera_cuota_pesos_fija_cop']:,.0f}",
        "¿Cumple Ley 546 (<=30%)?": "✅ Sí" if amort["primera_cuota_pesos_fija_cop"] <= tope_ley_546 else "❌ No",
        "Intereses Totales Pagados": f"${amort['total_intereses_pesos_fija_cop']:,.0f}",
        "Total Pagado (15 años)": f"${amort['total_pagado_pesos_fija_cop']:,.0f}",
    },
    {
        "Sistema SFC": "2. Pesos — Abono Constante a Capital",
        "Primera Cuota (COP)": f"${amort['primera_cuota_abono_constante_cop']:,.0f}",
        "¿Cumple Ley 546 (<=30%)?": "✅ Sí" if amort["primera_cuota_abono_constante_cop"] <= tope_ley_546 else "❌ No",
        "Intereses Totales Pagados": f"${amort['total_intereses_abono_constante_cop']:,.0f}",
        "Total Pagado (15 años)": f"${amort['total_pagado_abono_constante_cop']:,.0f}",
    },
    {
        "Sistema SFC": "3. UVR — Cuota Constante en UVR (IPC 5.2%)",
        "Primera Cuota (COP)": f"${amort['primera_cuota_uvr_cop']:,.0f}",
        "¿Cumple Ley 546 (<=30%)?": "✅ Sí" if amort["primera_cuota_uvr_cop"] <= tope_ley_546 else "❌ No",
        "Intereses Totales Pagados": f"${amort['total_intereses_uvr_cop']:,.0f}",
        "Total Pagado (15 años)": f"${amort['total_pagado_uvr_cop']:,.0f}",
    },
])

display(resumen_sistemas)
print("\n📌 Sensibilidad de la Cuota UVR en el Año 5 según Inflación IPC:")
for escenario, cuota_5 in amort["sensibilidad_cuota_uvr_anio_5"].items():
    print(f"   • {escenario}: ${cuota_5:,.0f} COP/mes")

In [ ]:
# Visualización comparativa: Evolución de la Cuota Mensual y del Saldo en Pesos (Fenómeno FA-01 en UVR)
fig_amort = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Evolución de la Cuota Mensual en COP vs. Tope Ley 546",
        "Evolución del Saldo de la Deuda en Pesos (Alerta FA-01 UVR)",
    ),
)

# Panel 1: Cuotas
fig_amort.add_trace(
    go.Scatter(x=df_pesos_fija["anio"], y=df_pesos_fija["cuota_total_cop"], name="Cuota Fija Pesos", line=dict(color="#1f77b4", width=3)),
    row=1, col=1,
)
fig_amort.add_trace(
    go.Scatter(x=df_pesos_const["anio"], y=df_pesos_const["cuota_total_cop"], name="Abono Constante Capital", line=dict(color="#2ca02c", width=2, dash="dot")),
    row=1, col=1,
)
fig_amort.add_trace(
    go.Scatter(x=df_uvr["anio"], y=df_uvr["cuota_total_cop"], name="Cuota en UVR (Indexada IPC)", line=dict(color="#d62728", width=3)),
    row=1, col=1,
)
fig_amort.add_hline(
    y=tope_ley_546,
    line_dash="dash",
    line_color="#7f7f7f",
    annotation_text=f"Tope 30% Ley 546 (${tope_ley_546:,.0f})",
    row=1, col=1,
)

# Panel 2: Saldos en COP
fig_amort.add_trace(
    go.Scatter(x=df_pesos_fija["anio"], y=df_pesos_fija["saldo_capital_cop"], name="Saldo Pesos Cuota Fija", line=dict(color="#1f77b4", width=3), showlegend=False),
    row=1, col=2,
)
fig_amort.add_trace(
    go.Scatter(x=df_pesos_const["anio"], y=df_pesos_const["saldo_capital_cop"], name="Saldo Abono Constante", line=dict(color="#2ca02c", width=2, dash="dot"), showlegend=False),
    row=1, col=2,
)
fig_amort.add_trace(
    go.Scatter(x=df_uvr["anio"], y=df_uvr["saldo_capital_cop"], name="Saldo UVR expresado en COP", line=dict(color="#d62728", width=3), showlegend=False),
    row=1, col=2,
)

fig_amort.update_layout(
    title="Comparativa Integral Ley 546 de 1999: Pesos Cuota Fija vs. Abono Constante vs. UVR",
    template="plotly_white",
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0),
)
fig_amort.update_xaxes(title_text="Año del Crédito")
fig_amort.update_yaxes(title_text="Pesos Colombianos (COP)")
fig_amort.show()

---
## 3. Paso 3: Analítica de Rentabilidad Inmobiliaria (*Cap Rate*, $VPN$, $TIR$ y Ley 820 de 2003)
Evaluamos el inmueble como activo generador de renta (*Buy-to-Rent*), respetando el tope legal del **1% mensual del valor comercial** establecido por el **Art. 18 de la Ley 820 de 2003**, descontando vacancia ($5\%$), predial, mantenimiento, administración y el impuesto de **Ganancia Ocasional ($15\%$, Ley 2277 de 2022)** en la liquidación al año 10, y lo contrastamos frente a **Arrendar e Invertir en un FIC Inmobiliario / Portafolio Financiero**.

In [ ]:
eval_maestra = evaluar_inversion_finca_raiz(
    valor_inmueble_cop=200_000_000.0,
    ingreso_familiar_neto_cop=9_415_000.0,
    porcentaje_cuota_inicial=0.30,
    plazo_entrega_planos_meses=24,
    cesantias_y_primas_anuales_cop=5_500_000.0,
    plazo_credito_anios=15,
    tasa_ea_pesos=0.115,
    tasa_ea_uvr_mas_spread=0.065,
    canon_arriendo_mensual_estimado_cop=1_450_000.0,
    costo_oportunidad_ea=0.105,
    ahorro_previo_afc_cop=8_000_000.0,
    config=cfg,
)

print("📊 INDICADORES FINANCIEROS DE INVERSIÓN INMOBILIARIA (CASO MAESTRO OED):")
print(f"   • Canon Mensual Aplicado (Ley 820 <= ${eval_maestra['tope_canon_ley_820_cop']:,.0f}): ${eval_maestra['canon_mensual_aplicado_cop']:,.0f} COP")
print(f"   • Ingreso Operativo Neto Anual (NOI): ${eval_maestra['noi_anual_cop']:,.0f} COP")
print(f"   • Cap Rate Bruto: {eval_maestra['cap_rate_bruto_pct']}% | Cap Rate Neto: {eval_maestra['cap_rate_neto_pct']}%")
print(f"   • Cash-on-Cash Año 1: {eval_maestra['cash_on_cash_anual_pct']}%")
print(f"   • TIR Inmobiliaria (10 años): {eval_maestra['tir_inmobiliaria_ea_pct']}% EA")
print(f"   • VPN descontado al COK (10.5% EA): ${eval_maestra['vpn_inmobiliario_cop']:,.0f} COP")
print(f"   • Dictamen Ejecutivo: {eval_maestra['dictamen_viabilidad_inmobiliaria']}")

comp = eval_maestra["comparativa_estrategias_10a"]
estrategias = [
    "1. Comprar en Pesos (Cuota Fija)",
    "2. Comprar en UVR (Indexado IPC)",
    "3. Arrendar + Invertir en FIC Inmobiliario",
]
patrimonios_10a = [
    comp["comprar_pesos_patrimonio_10a_cop"],
    comp["comprar_uvr_patrimonio_10a_cop"],
    comp["arrendar_e_invertir_fic_patrimonio_10a_cop"],
]

fig_comp = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Flujo de Caja Anual del Inversionista (Años 0 a 10)",
        "Patrimonio Neto Consolidado al Año 10 por Estrategia",
    ),
)

anios_flujo = list(range(0, 11))
flujos = eval_maestra["flujos_anuales_inversionista_cop"]
colores_flujo = ["#d62728" if f < 0 else "#2ca02c" for f in flujos]

fig_comp.add_trace(
    go.Bar(x=anios_flujo, y=flujos, marker_color=colores_flujo, name="Flujo Neto Anual (COP)"),
    row=1, col=1,
)
fig_comp.add_trace(
    go.Bar(
        x=estrategias,
        y=patrimonios_10a,
        marker_color=["#1f77b4", "#ff7f0e", "#9467bd"],
        text=[f"${p/1e6:,.1f}M" for p in patrimonios_10a],
        textposition="auto",
        name="Patrimonio Neto Año 10",
    ),
    row=1, col=2,
)

fig_comp.update_layout(
    title="Evaluación Financiera a 10 Años: Finca Raíz Directa vs. Alternativa FIC / Portafolio",
    template="plotly_white",
    height=460,
    showlegend=False,
)
fig_comp.update_xaxes(title_text="Año", row=1, col=1)
fig_comp.update_yaxes(title_text="Pesos Colombianos (COP)", row=1, col=1)
fig_comp.update_yaxes(title_text="Patrimonio Neto Año 10 (COP)", row=1, col=2)
fig_comp.show()

---
## 4. Simulador Interactivo de Jesi (Prueba Cualquier Apartamento Aquí)
Cambia cualquiera de los valores de abajo y ejecuta la celda para ver inmediatamente si el inmueble cumple la **Ley 546 de 1999**, cuánto falta de cuota inicial o ingreso en caso de que no cumpla, y cuál es su rentabilidad real.

In [ ]:
# ========= MODIFICA ESTOS PARÁMETROS PARA SIMULAR CUALQUIER CASO =========
VALOR_APARTAMENTO_COP = 280_000_000.0
INGRESO_FAMILIAR_NETO_COP = 6_500_000.0
PORCENTAJE_CUOTA_INICIAL = 0.30
PLAZO_PLANOS_MESES = 24
PLAZO_CREDITO_ANIOS = 15
MODALIDAD = "HIPOTECARIO"  # Prueba "HIPOTECARIO" o "LEASING"
# =========================================================================

mi_simulacion = evaluar_inversion_finca_raiz(
    valor_inmueble_cop=VALOR_APARTAMENTO_COP,
    ingreso_familiar_neto_cop=INGRESO_FAMILIAR_NETO_COP,
    porcentaje_cuota_inicial=PORCENTAJE_CUOTA_INICIAL,
    plazo_entrega_planos_meses=PLAZO_PLANOS_MESES,
    plazo_credito_anios=PLAZO_CREDITO_ANIOS,
    modalidad_financiacion=MODALIDAD,
    config=cfg,
)

print(f"🏠 Segmento: {mi_simulacion['segmento_vivienda']} ({mi_simulacion['valor_inmueble_smmlv']} SMMLV)")
print(f"💵 Cuota Inicial Mensual (sobre planos): ${mi_simulacion['cuota_inicial_mensual_cop']:,.0f} COP")
print(f"⚖️ Primera Cuota Pesos: ${mi_simulacion['primera_cuota_credito_cop']:,.0f} COP vs. Tope 30% (${mi_simulacion['tope_cuota_ley_546_30pct_cop']:,.0f} COP)")
print(f"🚦 ¿Cumple Ley 546 en Pesos?: {'✅ SÍ CUMPLE' if mi_simulacion['cumple_ley_546_30pct'] else '❌ NO CUMPLE'}")
if not mi_simulacion["cumple_ley_546_30pct"]:
    print(f"   👉 Solución A (Sumar ingresos en familia): +${mi_simulacion['deficit_ingreso_familiar_cop']:,.0f} COP/mes")
    print(f"   👉 Solución B (Aumentar cuota inicial):    +${mi_simulacion['cuota_inicial_extra_requerida_ley_546_cop']:,.0f} COP")
print(f"📈 Cap Rate Neto: {mi_simulacion['cap_rate_neto_pct']}% | TIR: {mi_simulacion['tir_inmobiliaria_ea_pct']}% EA")
if mi_simulacion["alertas_regulatorias"]:
    print("\n⚠️ Alertas Regulatorias Detectadas:")
    for alerta in mi_simulacion["alertas_regulatorias"]:
        print(f"   - {alerta}")

---
## 5. Integración Transversal OED (Módulos 1, 2, 3 y 6) y Voz de Asesor Financiero de Confianza
Validamos en vivo cómo el **Módulo 4 (Jesi)** conecta señales del resto del ecosistema OED:
- **Módulo 1 (Nidia):** Modalidad laboral (`Empleado`, `Independiente` / OPS o `Integral`) y protección preventiva del fondo de emergencia cubierto por **Fogafín**.
- **Módulo 3 (Catalina):** Cruce de cuotas de otras deudas vigentes frente al tope bancario total (**DTI $\le 40\%$**) además del **30% de la Ley 546**, recomendando el método **Avalancha**.
- **Módulo 6 (Salin) y Módulo 2 (Camilo):** Calibración automática del Costo de Oportunidad (**COK**) según el perfil de riesgo del inversionista (`Conservador`, `Moderado`, `Agresivo`).
- **Manual de Marca OED:** Traducción automática a **Diagnóstico Financiero** claro y **📌 Plan de Acción para este Mes**.

In [ ]:
cok_perfil = estimar_costo_oportunidad_por_perfil_oed(
    {"perfil_riesgo": "Moderado", "tasa_ponderada_esperada_ea_pct": 11.4},
    config=cfg,
)

eval_integrada = evaluar_inversion_finca_raiz(
    valor_inmueble_cop=200_000_000.0,
    ingreso_familiar_neto_cop=6_000_000.0,
    porcentaje_cuota_inicial=0.30,
    cesantias_y_primas_anuales_cop=0.0,
    costo_oportunidad_ea=cok_perfil["costo_oportunidad_ea"],
    ahorro_previo_afc_cop=7_000_000.0,
    cuotas_otras_deudas_mensuales_cop=1_400_000.0,
    fondo_emergencia_completo=False,
    modalidad_laboral_titular="Independiente",
    config=cfg,
)
diagnostico_asesor = generar_resumen_ejecutivo_ciudadano(eval_integrada)

print(f"🔗 COK calibrado desde Módulo 6: {cok_perfil['costo_oportunidad_ea']*100:.2f}% EA ({cok_perfil['fuente']})")
print(f"⚖️ Cumple 30% Ley 546 (solo vivienda): {eval_integrada['cumple_ley_546_30pct']}")
print(f"🏦 Cumple 40% Capacidad Bancaria Total (con otras deudas): {eval_integrada['cumple_endeudamiento_total_bancario_40pct']} ({eval_integrada['relacion_carga_financiera_total_pct']}%)")
print(f"🔓 Cupo mensual a liberar con Avalancha (Módulo 3): ${eval_integrada['cupo_mensual_a_liberar_deudas_cop']:,.0f} COP/mes")
print(f"\n💼 Diagnóstico Financiero ({diagnostico_asesor['nivel'].upper()}): {diagnostico_asesor['titulo']}")
print(f"   {diagnostico_asesor['mensaje']}")
print(f"📌 Plan de Acción para este Mes: {diagnostico_asesor['plan_accion']}")